# Layout (LSS) optimization
- single component optimization (as suggested by WEIS ppt)

##### references
1. 2020_Wang_NTNU - on design modelling and analysis of 10MW
2. IEA 15MW=baseline
3. Task2.1

imports

In [14]:
import numpy as np
import openmdao.api as om

In [15]:
from wisdem.drivetrainse.drivetrain import DriveMaterials

from wisdem.drivetrainse.hub import Hub_System
from wisdem.drivetrainse.gearbox import Gearbox

import wisdem.drivetrainse.layout as lay

import wisdem.drivetrainse.drive_components as dc

import wisdem.drivetrainse.drive_structure as ds

### Defining the model (problem)
as an openMDAO group that 
uses DrivetrainSE classes as components

In [50]:
class LSS_layout( om.Group ):
    """
    Group containing components for the layout of the LSS components
    """
    def initialize(self):
        self.options.declare("modeling_options")

    def setup(self):
        opt = self.options["modeling_options"]["WISDEM"]["DriveSE"]
        n_dlcs = self.options["modeling_options"]["WISDEM"]["n_dlc"]
        direct = opt["direct"]
        if direct:
            use_gb_torque_density = False
        else:
            use_gb_torque_density = opt["use_gb_torque_density"]
            
        dogen = self.options["modeling_options"]["flags"]["generator"]
        n_pc = self.options["modeling_options"]["WISDEM"]["RotorSE"]["n_pc"]
        flag_hub = self.options["modeling_options"]["flags"]["hub"]
        
        # print flag information
        print(f"flag info: direct={direct}, use_gb_torque_density={use_gb_torque_density}, dogen={dogen}, flag_hub={flag_hub}")

        # self.set_input_defaults("machine_rating", units="kW")
        #self.set_input_defaults("hvac_mass_coeff", 0.025, units="kg/kW/m")

        # Materials prep
        self.add_subsystem(
            "mat",
            DriveMaterials(direct=direct, n_mat=self.options["modeling_options"]["materials"]["n_mat"]),
                promotes=["*"]
            )
        # - for 'layout' component: need = lss_rho, bedplate_rho, hss_rho 

        # Before the layout, need to do these first
        # 1. hub system     
        if flag_hub: # NOTE: perf hub system optimization
            self.add_subsystem(
                "hub", Hub_System(modeling_options=opt["hub"]),
                    promotes=["*"]
                )
        
        # # 2. gearbox
        self.add_subsystem(
            "gear", Gearbox(direct_drive=direct, use_gb_torque_density=use_gb_torque_density),
                promotes=["*"]
            )

        # Layout
        #if not direct:
        self.add_subsystem(
            'layout', lay.GearedLayout(),
                promotes=["*"]
            )
        
        if False:
            # Hub_Rotor_LSS_Frame: components required =
            # 1. brake system
            self.add_subsystem(
                "brake", dc.Brake(direct_drive=direct),
                    promotes=["*"]
                )
            
            # 2. generator
            self.add_subsystem(
                "rpm", dc.RPM_Input(n_pc=n_pc),
                    promotes=["*"]
                )
            self.add_subsystem(
                "gensimp", dc.GeneratorSimple(direct_drive=direct, n_pc=n_pc),
                    promotes=["*"]
                )

            # Hub_Rotor_LSS_Frame:
            self.add_subsystem(
                "lss", ds.Hub_Rotor_LSS_Frame(n_dlcs=n_dlcs, modeling_options=opt, direct_drive=direct),
                    promotes=["*"]
                )
            
            # MBs for defl constraints
            self.add_subsystem("bear1", dc.MainBearing())
            self.add_subsystem("bear2", dc.MainBearing())

            # TODO: add HSS_Frame: perform HSS side optimization

            # Bedplate_IBeam_Frame:
            self.add_subsystem(
                "bed", ds.Bedplate_IBeam_Frame(modeling_options=opt, n_dlcs=n_dlcs),
                    promotes=["*"]
                )
            
            # NacelleSystemAdder: for nacelle_mass
            # - NOTE: yaw_mass default (0.0) used, coz dc.YawSystem not included (dc.Electronics also, for transformer_ and converter_mass)
            self.add_subsystem(
                "nac", dc.NacelleSystemAdder(direct_drive=direct),
                    promotes=["*"]
                )

            # Output-to-input connections
            # = mat -to- hub
            if flag_hub:
                self.connect("bedplate_rho", ["pitch_system.rho", "spinner.metal_rho"])
                self.connect("bedplate_Xy", ["pitch_system.Xy", "spinner.Xy"])
                self.connect("bedplate_mat_cost", "spinner.metal_cost")
                self.connect("hub_rho", "hub_shell.rho")
                self.connect("hub_Xy", "hub_shell.Xy")
                self.connect("hub_mat_cost", "hub_shell.metal_cost")
                self.connect("spinner_rho", "spinner.composite_rho")
                self.connect("spinner_Xt", "spinner.composite_Xt")
                self.connect("spinner_mat_cost", "spinner.composite_cost")

            # self.connect("hub_rho", "rho_castiron")
            # self.connect("spinner_rho", "rho_fiberglass")

            # = bearings -to- Bedplate_IBeam_Frame
            self.connect("bear1.mb_mass", "mb1_mass")
            self.connect("bear1.mb_I", "mb1_I")
            self.connect("bear1.mb_max_defl_ang", "mb1_max_defl_ang")
            self.connect("bear2.mb_mass", "mb2_mass")
            self.connect("bear2.mb_I", "mb2_I")
            self.connect("bear2.mb_max_defl_ang", "mb2_max_defl_ang")


define modelling_options

In [51]:
opt_flag = False

opts = {}

opts["WISDEM"] = {}
opts["WISDEM"]["n_dlc"] = 1
opts["WISDEM"]["DriveSE"] = {}
# NOTE "hub": 'Hub_System' component are NOT included in the 'LSS_layout' component 
opts["WISDEM"]["DriveSE"]["hub"] = {}
opts["WISDEM"]["DriveSE"]["hub"]["hub_gamma"] = 2.0
opts["WISDEM"]["DriveSE"]["hub"]["spinner_gamma"] = 1.5

opts["WISDEM"]["DriveSE"]["direct"] = False
opts["WISDEM"]["DriveSE"]["use_gb_torque_density"] = True # False =(GB  optim, in-capabale)

opts["WISDEM"]["DriveSE"]["gamma_f"] = 1.35 #IEC-1, 7.6.2.2a, pg.57
opts["WISDEM"]["DriveSE"]["gamma_m"] = 1.3  #IEC-1, 7.6.2.4, pg.59
opts["WISDEM"]["DriveSE"]["gamma_n"] = 1.0  #IEC-1, 7.6.1.3, pg.55
# used as: gamma = gamma_f * gamma_m * gamma_n (within TODO)

opts["WISDEM"]["RotorSE"] = {}
opts["WISDEM"]["RotorSE"]["n_pc"] = 2

opts["materials"] = {}
opts["materials"]["n_mat"] = 4

opts["flags"] = {}
opts["flags"]["generator"] = False
opts["flags"]["hub"] = False #(v)

### Setup the problem

In [52]:
# Define the problem
prob = om.Problem(reports=False)

# Define the model
prob.model = LSS_layout(modeling_options=opts) # an instance of the LSS_layout problem defined above

In [53]:
# If performing optimization, set up the optimizer and problem formulation
# TODO: Gearbox optimization ONLY ❗
if opt_flag:
    # Choose the optimizer to use
    prob.driver = om.ScipyOptimizeDriver()
    prob.driver.options["optimizer"] = "SLSQP"
    prob.driver.options["tol"] = 1e-2
    prob.driver.options["maxiter"] = 5 * 1

    # Add objective
    prob.model.add_objective("nacelle_mass", scaler=1e-6)

    # Add design variables
    prob.model.add_design_var("L_12", lower=0.1, upper=5.0)
    prob.model.add_design_var("L_h1", lower=0.1, upper=5.0)
    prob.model.add_design_var("lss_diameter", lower=0.5, upper=6.0)
    prob.model.add_design_var("lss_wall_thickness", lower=4e-3, upper=5e-1, ref=1e-2)

    # prob.model.add_design_var("L_hss", lower=0.1, upper=5.0)
    # prob.model.add_design_var("hss_diameter", lower=0.5, upper=6.0)
    # prob.model.add_design_var("hss_wall_thickness", lower=4e-3, upper=5e-1, ref=1e-2)

    prob.model.add_design_var("bedplate_web_thickness", lower=4e-3, upper=5e-1, ref=1e-2)
    prob.model.add_design_var("bedplate_flange_thickness", lower=4e-3, upper=5e-1, ref=1e-2)
    prob.model.add_design_var("bedplate_flange_width", lower=0.1, upper=2.0)

    # Add constraints on the tower design
    # 1. von Mises stress util
    prob.model.add_constraint("constr_lss_vonmises", upper=1.0)
    # prob.model.add_constraint("constr_hss_vonmises", upper=1.0) # HSS_Frame not included
    # prob.model.add_constraint("constr_bedplate_vonmises", upper=1.0)
    # 2. main bearing defl (allowed, max, as an angle)
    prob.model.add_constraint("constr_mb1_defl", upper=1.0)
    prob.model.add_constraint("constr_mb2_defl", upper=1.0)
    prob.model.add_constraint("constr_shaft_deflection", upper=1.0)
    prob.model.add_constraint("constr_shaft_angle", upper=1.0)
    # prob.model.add_constraint("constr_stator_deflection", upper=1.0)
    # prob.model.add_constraint("constr_stator_angle", upper=1.0)
    # 3. hub dia to accom. blades' roots
    #prob.model.add_constraint("constr_hub_diameter", lower=0.0)
    # 4. target overhang and hub height
    prob.model.add_constraint("constr_length", lower=0.0)
    prob.model.add_constraint("constr_height", lower=0.0)

In [54]:
# Setup the problem
prob.setup()

flag info: direct=False, use_gb_torque_density=True, dogen=False, flag_hub=False


In [55]:
print("All needed inputs to the model:")
for name, meta in prob.model.layout.list_inputs(out_stream=None, val=False):
    print(name)

All needed inputs to the model:
L_12
L_h1
L_generator
overhang
drive_height
tilt
lss_diameter
lss_wall_thickness
D_top
hub_diameter
lss_rho
bedplate_rho
bedplate_mass_user
L_hss
L_gearbox
hss_diameter
hss_wall_thickness
hss_rho
bedplate_flange_width
bedplate_flange_thickness
bedplate_web_thickness
upwind


### Defining input values
after calling prob.setup() (on the openMDAO 'prob' defined) and before calling prob.run_driver()

1. High-level Inputs

In [56]:
prob.set_val("machine_rating", 15.0, units="MW")
prob["rotor_diameter"] = 240.0
prob["rated_torque"] = 4308926.79641971

prob["upwind"] = True
prob["D_top"] = 6.5 #tower top diameter
# prob["minimum_rpm"] = 5
# prob["rated_rpm"] = 7.56
# prob["hub_diameter"] = 7.94
prob["overhang"] = 11.35 #ref.2
prob["tilt"] = 6.0 #ref.3

# TODO: replace hub.py and user input hub inputs for LSS
# RNA_mass = 1017 (tons) (cf. tab. ES-1, ref.2)
# nacelle mass = 820.888 (tons) (cf. tab. 5-1, ref.2)
# nacelle mass minus hub = 630.888 (tons)
# so, hub mass = 190 (tons)

# Loading from rotor (TODO: actual ULS loads)
# prob["F_aero_hub"] = np.array([1125044.07614847, -7098.0872533, -7022.79756034]).reshape((3, 1))
# prob["M_aero_hub"] = np.array([10515165.10636333, 945938.60268626, 1042828.16100417]).reshape((3, 1))
# ---

2. Blade properties and hub design options
- cf. `opts["flags"]["hub"]`

In [57]:
# Hub_Rotor_LSS_Frame inputs
# TODO: copied from made4wind_geared (IEA-15MW = ref), change to made4wind specs
if False:
    blade_mass = 65252.0
    n_blades = 3
    prob["blades_mass"] = n_blades * blade_mass
    prob["blades_cm"] = 2.46175
    prob["blades_I"] = np.r_[3.48453857e+08, 1.74226928e+08, 1.74226928e+08, np.zeros(3)]

    # if run HUB module within DriveSE
    if opts["flags"]["hub"]:
        prob["flange_t2shell_t"] = 6.0
        prob["flange_OD2hub_D"] = 0.6
        prob["flange_ID2flange_OD"] = 0.8
        prob["hub_in2out_circ"] = 1.2
        prob["hub_stress_concentration"] = 3.0
        prob["n_front_brackets"] = 5
        prob["n_rear_brackets"] = 5
        prob["clearance_hub_spinner"] = 0.5
        prob["spin_hole_incr"] = 1.2
        prob["blade_root_diameter"] = 5.2

        prob["n_blades"] = 3
        prob["blade_mass"] = 65252.0
        prob["blades_mass"] = prob["n_blades"] * prob["blade_mass"]
        prob["blades_cm"] = 2.46175
        prob["blades_I"] = np.r_[3.48453857e+08, 1.74226928e+08, 1.74226928e+08, np.zeros(3)]

        prob["pitch_system.BRFM"] = 26648449.0
        prob["pitch_system_scaling_factor"] = 0.75

        prob["spinner_gust_ws"] = 70.0

    else:
        # run made4wind_geared.py with opt_flag = false and copy the following values from drivetrain_example.csv
        prob["hub_system_mass"] = 62561.91718921
        prob["hub_system_cm"] = 3.35947759
        prob["hub_system_I"] = np.array([[865503.52531197, 567289.77714803, 567289.77714803],[0., 0., 0.]])

3. Drivetrain configuration and sizing inputs

In [58]:
myones = np.ones(2)
 
# - init condn for some design vars
# prob["bear1.bearing_type"] = "SRB" # 1. fixed MB
# prob["bear2.bearing_type"] = "CARB" # 2. floating MB
# prob["bear1.D_shaft"] = 2.2
# prob["bear2.D_shaft"] = 2.2

prob["L_h1"] = 1.0
prob["L_12"] = 1.2
prob["lss_diameter"] = 1.0 * myones
prob["lss_wall_thickness"] = 0.288 * myones

# Gearbox inputs
# prob["L_gearbox"] = 1.5 #(v) calc in gearbox.py
prob["gear_configuration"] = "eee"
prob["planet_numbers"] = np.array([5, 3, 0]) #ref.1
prob["gear_ratio"] = 50 #.039
prob["gearbox_mass_user"] = 0.0 #(cf. line 206, gearbox.py)
prob["gearbox_torque_density"] = 200.0 # (cf. line 210, gearbox.py)

prob["L_hss"] = 1.5
prob["hss_diameter"] = 0.5 * myones
prob["hss_wall_thickness"] = 0.1 * myones
prob["L_generator"] = 2.15

# needed by Bedplate_IBeam_Frame in layout.py, output of HSS_Frame
# copied from made4wind_geared.py's output drivetrain_example.csv
# prob["F_generator"] = np.array([[-48863.264001575044], [-0.0], [-558509.6649410343]])
# prob["M_generator"] = np.array([[420611.22], [-2713550.7624924504], [-0.0]])

prob["bedplate_flange_width"] = 1.0
prob["bedplate_flange_thickness"] = 0.1
prob["bedplate_web_thickness"] = 0.1

# 'drive_height' : derive from the high-level inputs (output= 5.95522 m)
# - needed by layout.py (line 122)
L_fl = 0.358 #ref.2: Hub flange length 
L2n = 0.9 #ref.2: Distance of downwind bearing from bedplate flange
L_lss = prob["L_h1"]+prob["L_12"]+L2n
H_nose = 4.875 #ref.2: Nose height (from tower top to bottom of bedplate flange)
prob["drive_height"] = H_nose + ( np.sin(prob["tilt"]*np.pi/180)*( (prob["hub_diameter"]*np.sqrt(3/4))+L_fl+L_lss ) ) # = 5.95522 m

# prob["shaft_deflection_allowable"] = 1e-4 # within Hub_Rotor_LSS_Frame (below): Deflections and rotations at GB attachment
# prob["shaft_angle_allowable"] = 1e-3
# prob["stator_deflection_allowable"] = 1e-4 # within Bedplate_IBeam_Frame (below)
# prob["stator_angle_allowable"] = 1e-3

In [59]:
print( prob["drive_height"] )

[5.23645943]


4. Material properties (discrete_inputs to DriveMaterials)

In [60]:
# DriveMaterials inputs
prob["E_mat"] = np.c_[200e9 * np.ones(3), 205e9 * np.ones(3), 118e9 * np.ones(3), [4.46e10, 1.7e10, 1.67e10]].T
prob["G_mat"] = np.c_[79.3e9 * np.ones(3), 80e9 * np.ones(3), 47.6e9 * np.ones(3), [3.27e9, 3.48e9, 3.5e9]].T
prob["Xt_mat"] = np.c_[450e6 * np.ones(3), 814e6 * np.ones(3), 310e6 * np.ones(3), [6.092e8, 3.81e7, 1.529e7]].T
# - (v, note) these would be  -np.c_-> (4,3) -.T-> (3,4) array
prob["rho_mat"] = np.r_[7800.0, 7850.0, 7200.0, 1940.0]
prob["Xy_mat"] = np.r_[345e6, 485e6, 265e6, 18.9e6]
prob["wohler_exp_mat"] = 1e1 * np.ones(4)
prob["wohler_A_mat"] = 1e1 * np.ones(4)
prob["unit_cost_mat"] = np.r_[0.7, 0.9, 0.5, 1.9]
# - Material assignment
prob["lss_material"] = prob["hss_material"] = "steel_drive"
prob["bedplate_material"] = "steel"
prob["hub_material"] = "cast_iron"
prob["spinner_material"] = "glass_uni"
prob["material_names"] = ["steel", "steel_drive", "cast_iron", "glass_uni"]
# ---

### Running
_model (analysis) or _driver (optimization)

In [ ]:
if opt_flag:
    # set up the optimization and problem formulation
    # NOTE: objective "nacelle_mass" = 409215.9885880007, from drivetrain_example.csv

    # Run the optimization
    prob.model.approx_totals()
    prob.run_driver()
else:
    # Run the analysis
    prob.run_model()

# Print the results: TODO
print('=== outputs ===:')
print( prob.model.list_inputs() )
print( prob.model.list_outputs() )

for output_name in prob.model.list_outputs(out_stream=None):
    print(f'{ output_name[0] }: { prob.get_val(output_name[0]) }')

=== outputs ===:
46 Input(s) in 'model'

varname                      val                                                 prom_name                
---------------------------  --------------------------------------------------  -------------------------
mat
  E_mat                      |538891501139.144|                                  E_mat                    
  G_mat                      |211891017506.64185|                                G_mat                    
  Xt_mat                     |1804549260.6465473|                                Xt_mat                   
  Xy_mat                     |651791538.7606685|                                 Xy_mat                   
  wohler_exp_mat             |20.0|                                              wohler_exp_mat           
  wohler_A_mat               |20.0|                                              wohler_A_mat             
  rho_mat                    |13344.14103642|                                    rho_mat           

: 

In [62]:
prob.model.gear.list_inputs()
prob.model.gear.list_outputs()

10 Input(s) in 'gear'

varname                 val                  prom_name             
----------------------  -------------------  ----------------------
gear_ratio              [50.]                gear_ratio            
rotor_diameter          [240.]               rotor_diameter        
rated_torque            [4308926.79641971]   rated_torque          
machine_rating          [15000.]             machine_rating        
gearbox_mass_user       [0.]                 gearbox_mass_user     
gearbox_torque_density  [200.]               gearbox_torque_density
gearbox_radius_user     [0.]                 gearbox_radius_user   
gearbox_length_user     [0.]                 gearbox_length_user   
gear_configuration      eee                  gear_configuration    
planet_numbers          |5.83095189|         planet_numbers        


7 Explicit Output(s) in 'gear'

varname       val                   prom_name   
------------  --------------------  ------------
stage_ratios  |0.0|          

[('stage_ratios', {'val': array([0., 0., 0.]), 'prom_name': 'stage_ratios'}),
 ('gearbox_mass',
  {'val': array([21544.6339821]), 'prom_name': 'gearbox_mass'}),
 ('gearbox_I',
  {'val': array([22337.47651264, 34436.94295699, 34436.94295699]),
   'prom_name': 'gearbox_I'}),
 ('L_gearbox', {'val': array([3.6]), 'prom_name': 'L_gearbox'}),
 ('D_gearbox', {'val': array([2.88]), 'prom_name': 'D_gearbox'}),
 ('carrier_mass', {'val': array([13000.]), 'prom_name': 'carrier_mass'}),
 ('carrier_I',
  {'val': array([13478.4,  6739.2,  6739.2]), 'prom_name': 'carrier_I'})]